# TenueCheck - Entrainement YOLOv8n (couvre_chef)

**Pipeline :** Collecte -> Nettoyage -> Exploitation -> Modelisation -> Evaluation -> Production

| Classe | ID | Description |
|---|---|---|
| couvre_chef | 0 | Casquette, chapeau, bonnet, capuche |

**IMPORTANT** : `Execution > Modifier le type d'execution` -> **GPU T4**

---
## 0. Environnement

In [ ]:
!pip install -q ultralytics datasets huggingface_hub tqdm

import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("ATTENTION: pas de GPU ! Allez dans Execution > Modifier le type d'execution > GPU T4")

---
## 1. COLLECTE - Telechargement des donnees

On telecharge le dataset **Fashionpedia** depuis HuggingFace (~45000 images de mode annotees).
On filtre uniquement les images contenant des couvre-chefs (hat, headband, hood).
On utilise le **streaming** pour eviter de charger tout le dataset en RAM.

In [ ]:
import os
import gc
import shutil
import random
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from datasets import load_dataset

# === CONFIGURATION ===
TARGET_CLASSES = {0: "couvre_chef"}

# Mapping Fashionpedia -> notre classe
FASHIONPEDIA_TO_TARGET = {
    14: 0,  # hat -> couvre_chef
    15: 0,  # headband, head covering -> couvre_chef
    27: 0,  # hood -> couvre_chef
}

OUTPUT_DIR = "dataset_tenuecheck"
IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
LABELS_DIR = os.path.join(OUTPUT_DIR, "labels")
MAX_IMAGES = 2000

# Creer l'arborescence
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(IMAGES_DIR, split), exist_ok=True)
    os.makedirs(os.path.join(LABELS_DIR, split), exist_ok=True)

# Streaming = pas de telechargement complet en RAM
print("Chargement de Fashionpedia en streaming...")
ds = load_dataset("detection-datasets/fashionpedia", split="train", streaming=True)
print("Dataset pret (mode streaming)")

In [ ]:
# Filtrer et sauvegarder directement (sans stocker en RAM)
target_categories = set(FASHIONPEDIA_TO_TARGET.keys())
saved = []
skipped = 0
total_seen = 0

for sample in tqdm(ds, desc="Collecte + conversion YOLO", total=MAX_IMAGES):
    total_seen += 1
    categories = sample["objects"]["category"]

    # Verifier si l'image contient au moins un couvre-chef
    if not any(cat in target_categories for cat in categories):
        continue

    image = sample["image"]
    objects = sample["objects"]
    img_w, img_h = image.size

    yolo_annotations = []
    for cat, bbox in zip(objects["category"], objects["bbox"]):
        if cat not in target_categories:
            continue
        target_class = FASHIONPEDIA_TO_TARGET[cat]
        # COCO [x_min, y_min, w, h] -> YOLO [x_center, y_center, w, h] normalise
        x_min, y_min, w, h = bbox
        x_center = max(0, min(1, (x_min + w / 2) / img_w))
        y_center = max(0, min(1, (y_min + h / 2) / img_h))
        w_norm = max(0, min(1, w / img_w))
        h_norm = max(0, min(1, h / img_h))
        if w_norm > 0.01 and h_norm > 0.01:
            yolo_annotations.append(f"{target_class} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

    if yolo_annotations:
        i = len(saved)
        img_filename = f"fp_{i:05d}.jpg"
        image.convert("RGB").save(os.path.join(IMAGES_DIR, "train", img_filename), "JPEG", quality=95)
        with open(os.path.join(LABELS_DIR, "train", f"fp_{i:05d}.txt"), "w") as f:
            f.write("\n".join(yolo_annotations))
        saved.append(img_filename)
    else:
        skipped += 1

    if len(saved) >= MAX_IMAGES:
        break

# Liberer la RAM
del ds
gc.collect()

print(f"\n{len(saved)} images sauvegardees | {skipped} ignorees | {total_seen} images parcourues")

---
## 2. NETTOYAGE - Validation et qualite des donnees

On verifie que les images ne sont pas corrompues et que les annotations sont valides.
Les echantillons problematiques sont supprimes automatiquement.

In [ ]:
import cv2
import numpy as np

train_img_dir = os.path.join(IMAGES_DIR, "train")
train_lbl_dir = os.path.join(LABELS_DIR, "train")

corrupted = []
bad_labels = []
empty_labels = []
valid = 0

all_images = [f for f in os.listdir(train_img_dir) if f.endswith(".jpg")]

for img_file in tqdm(all_images, desc="Verification"):
    img_path = os.path.join(train_img_dir, img_file)
    lbl_path = os.path.join(train_lbl_dir, img_file.replace(".jpg", ".txt"))

    # Verifier que l'image est lisible
    try:
        img = Image.open(img_path)
        img.verify()
    except Exception:
        corrupted.append(img_file)
        continue

    # Verifier que le label existe
    if not os.path.exists(lbl_path):
        empty_labels.append(img_file)
        continue

    # Verifier que le label est valide (5 valeurs par ligne, coords entre 0 et 1)
    with open(lbl_path) as f:
        lines = f.readlines()

    is_valid = True
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            is_valid = False
            break
        cls_id = int(parts[0])
        coords = [float(x) for x in parts[1:]]
        if cls_id not in TARGET_CLASSES or any(c < 0 or c > 1 for c in coords):
            is_valid = False
            break

    if not is_valid:
        bad_labels.append(img_file)
    else:
        valid += 1

# Supprimer les fichiers problematiques
removed = 0
for img_file in corrupted + bad_labels + empty_labels:
    img_path = os.path.join(train_img_dir, img_file)
    lbl_path = os.path.join(train_lbl_dir, img_file.replace(".jpg", ".txt"))
    if os.path.exists(img_path):
        os.remove(img_path)
    if os.path.exists(lbl_path):
        os.remove(lbl_path)
    removed += 1

print(f"\n--- Rapport de nettoyage ---")
print(f"  Images valides:     {valid}")
print(f"  Images corrompues:  {len(corrupted)}")
print(f"  Labels invalides:   {len(bad_labels)}")
print(f"  Labels manquants:   {len(empty_labels)}")
print(f"  Total supprime:     {removed}")
print(f"  Dataset nettoye:    {valid} images pretes")

---
## 3. EXPLOITATION - Analyse exploratoire

Distribution des classes, tailles des bounding boxes, et visualisation d'echantillons.

In [ ]:
import matplotlib.pyplot as plt

# Compter les annotations par classe
class_counts = {name: 0 for name in TARGET_CLASSES.values()}
bbox_sizes = {name: [] for name in TARGET_CLASSES.values()}

for lbl_file in os.listdir(train_lbl_dir):
    if not lbl_file.endswith(".txt"):
        continue
    with open(os.path.join(train_lbl_dir, lbl_file)) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id = int(parts[0])
                w, h = float(parts[3]), float(parts[4])
                cls_name = TARGET_CLASSES.get(cls_id, "unknown")
                if cls_name in class_counts:
                    class_counts[cls_name] += 1
                    bbox_sizes[cls_name].append(w * h)

# --- Graphique 1 : Distribution des classes ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(class_counts.keys())
counts = list(class_counts.values())

axes[0].bar(names, counts, color='#FF6B35')
axes[0].set_title("Distribution des classes", fontsize=14)
axes[0].set_ylabel("Nombre d'annotations")
for i, (name, count) in enumerate(zip(names, counts)):
    axes[0].text(i, count + 10, str(count), ha='center', fontweight='bold')

# --- Graphique 2 : Tailles des bounding boxes ---
for cls_name, sizes in bbox_sizes.items():
    if sizes:
        axes[1].hist(sizes, bins=30, alpha=0.6, label=cls_name, color='#FF6B35')
axes[1].set_title("Distribution des tailles de bbox", fontsize=14)
axes[1].set_xlabel("Aire normalisee (w * h)")
axes[1].set_ylabel("Frequence")
axes[1].legend()

plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=150)
plt.show()

print(f"\nTotal annotations: {sum(counts)}")
for name, count in zip(names, counts):
    pct = count/sum(counts)*100 if sum(counts) > 0 else 0
    print(f"  {name}: {count} ({pct:.1f}%)")

In [ ]:
# Visualiser des echantillons avec leurs annotations
import cv2
import numpy as np
from IPython.display import display, Image as IPImage

sample_images = [f for f in os.listdir(train_img_dir) if f.endswith(".jpg")][:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, img_file in enumerate(sample_images):
    img = cv2.imread(os.path.join(train_img_dir, img_file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    lbl_path = os.path.join(train_lbl_dir, img_file.replace(".jpg", ".txt"))
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                xc, yc, bw, bh = [float(x) for x in parts[1:]]
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                color = (255, 106, 53)  # orange
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, TARGET_CLASSES[cls_id], (x1, y1-5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    axes[i].imshow(img)
    axes[i].set_title(img_file, fontsize=8)
    axes[i].axis('off')

plt.suptitle("Echantillons du dataset avec annotations", fontsize=16)
plt.tight_layout()
plt.savefig("eda_samples.png", dpi=150)
plt.show()

---
## 4. MODELISATION - Split et entrainement YOLOv8n

**Transfer learning** : on charge `yolov8n.pt` (pre-entraine sur COCO, 80 classes)
et on affine (fine-tune) sur notre classe couvre_chef.

In [ ]:
# === SPLIT TRAIN / VAL / TEST ===
TRAIN_RATIO, VAL_RATIO = 0.8, 0.15

all_images = [f for f in os.listdir(os.path.join(IMAGES_DIR, "train")) if f.endswith(".jpg")]
random.shuffle(all_images)

n = len(all_images)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

splits = {
    "val": all_images[n_train:n_train + n_val],
    "test": all_images[n_train + n_val:]
}

for split, files in splits.items():
    for img_file in files:
        label_file = img_file.rsplit(".", 1)[0] + ".txt"
        for src_dir, dst_dir in [(IMAGES_DIR, IMAGES_DIR), (LABELS_DIR, LABELS_DIR)]:
            src = os.path.join(src_dir, "train", img_file if src_dir == IMAGES_DIR else label_file)
            dst = os.path.join(dst_dir, split, img_file if src_dir == IMAGES_DIR else label_file)
            if os.path.exists(src):
                shutil.move(src, dst)

train_count = len(os.listdir(os.path.join(IMAGES_DIR, "train")))
val_count = len(os.listdir(os.path.join(IMAGES_DIR, "val")))
test_count = len(os.listdir(os.path.join(IMAGES_DIR, "test")))
print(f"Train: {train_count} | Val: {val_count} | Test: {test_count}")

In [ ]:
# === FICHIER YAML ===
yaml_content = f"""# TenueCheck Dataset
path: {os.path.abspath(OUTPUT_DIR)}
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: couvre_chef
"""

yaml_path = os.path.join(OUTPUT_DIR, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"YAML cree: {yaml_path}")

In [ ]:
# === ENTRAINEMENT YOLOv8n ===
from ultralytics import YOLO

# Transfer learning : poids pre-entraines COCO
model = YOLO("yolov8n.pt")

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name="tenuecheck_yolov8n",
    patience=15,
    save=True,
    plots=True,
    device=0,
    workers=2,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.15,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

---
## 5. EVALUATION - Metriques et analyse des performances

In [ ]:
# Courbes d'entrainement et matrice de confusion
results_dir = "runs/detect/tenuecheck_yolov8n"

for img_name in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png"]:
    img_path = os.path.join(results_dir, img_name)
    if os.path.exists(img_path):
        print(f"\n--- {img_name} ---")
        display(IPImage(filename=img_path, width=800))

In [ ]:
# Evaluation sur le jeu de test
best_model = YOLO(os.path.join(results_dir, "weights", "best.pt"))
metrics = best_model.val(data=yaml_path, split="test", verbose=False)

print("=" * 50)
print("METRIQUES SUR LE JEU DE TEST")
print("=" * 50)
print(f"  mAP50:      {metrics.box.map50:.3f}")
print(f"  mAP50-95:   {metrics.box.map:.3f}")
print(f"  Precision:  {metrics.box.mp:.3f}")
print(f"  Recall:     {metrics.box.mr:.3f}")
print(f"\nPar classe :")
for i, cls_name in TARGET_CLASSES.items():
    if i < len(metrics.box.ap50):
        print(f"  {cls_name:15s} -> AP50: {metrics.box.ap50[i]:.3f}")

In [ ]:
# Test visuel sur des images du jeu de test
import glob

test_images = glob.glob(os.path.join(IMAGES_DIR, "test", "*.jpg"))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, img_path in enumerate(test_images):
    results = best_model(img_path, conf=0.5, verbose=False)
    for r in results:
        annotated = r.plot()
        axes[i].imshow(annotated[..., ::-1])
        n_det = len(r.boxes)
        det_names = [TARGET_CLASSES.get(int(b.cls[0]), "?") for b in r.boxes]
        title = f"{n_det} detection(s): {', '.join(det_names)}" if n_det > 0 else "Aucune detection"
        axes[i].set_title(title, fontsize=10)
        axes[i].axis('off')

plt.suptitle("Predictions sur le jeu de TEST", fontsize=16)
plt.tight_layout()
plt.savefig("eval_test_predictions.png", dpi=150)
plt.show()

In [ ]:
# Impact du seuil de confiance
print("Impact du seuil de confiance sur le jeu de test :\n")
print(f"{'Seuil':>8s} | {'mAP50':>8s} | {'Precision':>10s} | {'Recall':>8s}")
print("-" * 45)

for conf in [0.3, 0.5, 0.7, 0.8]:
    m = best_model.val(data=yaml_path, split="test", conf=conf, verbose=False)
    print(f"{conf:>8.1f} | {m.box.map50:>8.3f} | {m.box.mp:>10.3f} | {m.box.mr:>8.3f}")

print("\n-> Choisissez le seuil qui donne le meilleur equilibre precision/recall.")

---
## 6. PRODUCTION - Export et deploiement

Telechargez `dresscode_yolo.pt` et placez-le a la racine de votre projet TenueCheck.
Lancez ensuite votre application avec la camera.

In [ ]:
# Copier le meilleur modele
best_model_path = Path(os.path.join(results_dir, "weights", "best.pt"))
if best_model_path.exists():
    shutil.copy(best_model_path, "dresscode_yolo.pt")
    size_mb = best_model_path.stat().st_size / 1024 / 1024
    print(f"Modele copie: dresscode_yolo.pt ({size_mb:.1f} MB)")
else:
    for p in Path("runs/detect").glob("*/weights/best.pt"):
        shutil.copy(p, "dresscode_yolo.pt")
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"Modele copie depuis {p}")
        break

# Resume final
print(f"\n{'='*50}")
print("RESUME DU PIPELINE")
print(f"{'='*50}")
print(f"  Collecte:      {len(saved)} images telechargees")
print(f"  Dataset final: {train_count} train / {val_count} val / {test_count} test")
print(f"  Classe:        couvre_chef")
print(f"  Modele:        YOLOv8n (transfer learning COCO)")
print(f"  mAP50:         {metrics.box.map50:.3f}")
print(f"  Precision:     {metrics.box.mp:.3f}")
print(f"  Recall:        {metrics.box.mr:.3f}")
print(f"  Fichier:       dresscode_yolo.pt")

In [ ]:
# Telecharger le modele
from google.colab import files
files.download("dresscode_yolo.pt")
print("Telechargement lance ! Placez le fichier a la racine de votre projet.")